# 03 - Anomaly Detection
## Isolation Forest, Z-Score, and IQR Methods

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.terna_loader import load_demand_data, load_frequency_data
from src.models.ml_models import detect_anomalies_isolation_forest, detect_anomalies_zscore, detect_anomalies_iqr

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## Load Data

In [ ]:
demand = load_demand_data()
frequency = load_frequency_data()
print(f'Demand: {demand.shape}, Frequency: {frequency.shape}')

## Method 1: Isolation Forest on Demand

In [ ]:
demand_features = demand.copy()
demand_features['hour'] = demand_features.index.hour
demand_features['dayofweek'] = demand_features.index.dayofweek
demand_features['demand_lag_24'] = demand_features['demand_mw'].shift(24)
demand_features['demand_rolling_24'] = demand_features['demand_mw'].rolling(24).mean()
demand_features = demand_features.dropna()

features = ['demand_mw', 'hour', 'dayofweek', 'demand_lag_24', 'demand_rolling_24']

result_if, model_if = detect_anomalies_isolation_forest(
    demand_features, features, contamination=0.01
)

n_anomalies = result_if['anomaly'].sum()
print(f'Isolation Forest: {n_anomalies} anomalies detected ({n_anomalies/len(result_if)*100:.2f}%)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(result_if.index, result_if['demand_mw'], linewidth=0.5, alpha=0.7)
anomaly_points = result_if[result_if['anomaly']]
axes[0].scatter(anomaly_points.index, anomaly_points['demand_mw'], 
                color='red', s=10, label='Anomaly', zorder=5)
axes[0].set_title('Demand with Anomalies (Isolation Forest)')
axes[0].set_ylabel('Demand (MW)')
axes[0].legend()

axes[1].plot(result_if.index, result_if['anomaly_score'], linewidth=0.5, alpha=0.7)
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].fill_between(result_if.index, result_if['anomaly_score'], 0, 
                     where=result_if['anomaly_score'] < 0, alpha=0.3, color='red')
axes[1].set_title('Anomaly Score')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

## Method 2: Z-Score on Demand

In [ ]:
result_zscore = detect_anomalies_zscore(demand, 'demand_mw', threshold=3.0)

n_anomalies_z = result_zscore['anomaly'].sum()
print(f'Z-Score: {n_anomalies_z} anomalies detected ({n_anomalies_z/len(result_zscore)*100:.2f}%)')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(result_zscore.index, result_zscore['demand_mw'], linewidth=0.5, alpha=0.7)
anomaly_z = result_zscore[result_zscore['anomaly']]
axes[0].scatter(anomaly_z.index, anomaly_z['demand_mw'], 
                color='red', s=10, label='Anomaly', zorder=5)
axes[0].set_title('Demand with Anomalies (Z-Score > 3)')
axes[0].set_ylabel('Demand (MW)')
axes[0].legend()

axes[1].plot(result_zscore.index, result_zscore['zscore'], linewidth=0.5, alpha=0.7)
axes[1].axhline(y=3, color='r', linestyle='--', alpha=0.5)
axes[1].axhline(y=-3, color='r', linestyle='--', alpha=0.5)
axes[1].set_title('Z-Score')
axes[1].set_ylabel('Z-Score')

plt.tight_layout()
plt.show()

## Method 3: IQR on Demand

In [ ]:
result_iqr = detect_anomalies_iqr(demand, 'demand_mw', multiplier=1.5)

n_anomalies_iqr = result_iqr['anomaly'].sum()
print(f'IQR: {n_anomalies_iqr} anomalies detected ({n_anomalies_iqr/len(result_iqr)*100:.2f}%)')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(result_iqr.index, result_iqr['demand_mw'], linewidth=0.5, alpha=0.7)
ax.axhline(y=result_iqr['anomaly_lower'].iloc[0], color='r', linestyle='--', alpha=0.5, label='IQR Bounds')
ax.axhline(y=result_iqr['anomaly_upper'].iloc[0], color='r', linestyle='--', alpha=0.5)
anomaly_iqr = result_iqr[result_iqr['anomaly']]
ax.scatter(anomaly_iqr.index, anomaly_iqr['demand_mw'], 
           color='red', s=10, label='Anomaly', zorder=5)
ax.set_title('Demand with Anomalies (IQR Method)')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.show()

## Frequency Anomaly Detection

In [ ]:
result_freq = detect_anomalies_zscore(frequency, 'frequency_hz', threshold=3.0)

n_anomalies_freq = result_freq['anomaly'].sum()
print(f'Frequency anomalies: {n_anomalies_freq} ({n_anomalies_freq/len(result_freq)*100:.2f}%)')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(result_freq.index, result_freq['frequency_hz'], linewidth=0.5, alpha=0.7)
anomaly_f = result_freq[result_freq['anomaly']]
ax.scatter(anomaly_f.index, anomaly_f['frequency_hz'], 
           color='red', s=10, label='Anomaly', zorder=5)
ax.axhline(y=50.0, color='green', linestyle='--', alpha=0.5, label='Nominal 50 Hz')
ax.set_title('Grid Frequency with Anomalies')
ax.set_ylabel('Frequency (Hz)')
ax.legend()
plt.tight_layout()
plt.show()

## Method Comparison

In [ ]:
comparison = pd.DataFrame({
    'Method': ['Isolation Forest', 'Z-Score', 'IQR'],
    'Anomalies': [n_anomalies, n_anomalies_z, n_anomalies_iqr],
    'Rate (%)': [
        n_anomalies/len(result_if)*100,
        n_anomalies_z/len(result_zscore)*100,
        n_anomalies_iqr/len(result_iqr)*100,
    ]
})
print(comparison.to_string(index=False))

In [ ]:
print('Notebook 03 complete. Anomaly detection analysis done.')